# Healthcare EMR Appointment Data - Enhanced ML Preprocessing Pipeline
## Goal: Appointment Slot Recommendation System

In [15]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [16]:
df = pd.read_csv("../dataset/raw/full_appointments.csv")
print(f"Dataset Shape: {df.shape}")

Dataset Shape: (1000, 81)


## 2. Parse All DateTime Columns

In [17]:
df['dob'] = pd.to_datetime(df['dob'], format='%m/%d/%Y', errors='coerce')
df['startDate'] = pd.to_datetime(df['startDate'], errors='coerce').dt.tz_localize(None)
df['endDate'] = pd.to_datetime(df['endDate'], errors='coerce').dt.tz_localize(None)
df['createOn'] = pd.to_datetime(df['createOn'], errors='coerce').dt.tz_localize(None)
df['updatedOn'] = pd.to_datetime(df['updatedOn'], errors='coerce').dt.tz_localize(None)
df['lastPaymentDate'] = pd.to_datetime(df['lastPaymentDate'], errors='coerce')
df['insurance_coverageStartDate'] = pd.to_datetime(df['insurance_coverageStartDate'], format='%m/%d/%Y', errors='coerce')
df['secondaryInsurance_coverageStartDate'] = pd.to_datetime(df['secondaryInsurance_coverageStartDate'], format='%m/%d/%Y', errors='coerce')
df['secondaryInsurance_terminationDate'] = pd.to_datetime(df['secondaryInsurance_terminationDate'], format='%m/%d/%Y', errors='coerce')
print("DateTime parsing completed")

DateTime parsing completed


## 3. Temporal Feature Engineering + Holidays & Peak Days

In [18]:
# Patient age
df['patient_age'] = ((df['startDate'] - df['dob']).dt.days / 365.25).round(0)

# Temporal features
df['appt_hour'] = df['startDate'].dt.hour
df['appt_weekday'] = df['startDate'].dt.dayofweek
df['appt_month'] = df['startDate'].dt.month
df['appt_day'] = df['startDate'].dt.day
df['appt_quarter'] = df['startDate'].dt.quarter
df['is_weekend'] = (df['appt_weekday'] >= 5).astype(int)

# Time slot
df['time_slot'] = pd.cut(df['appt_hour'], bins=[0, 9, 12, 15, 24], labels=['Morning', 'Midday', 'Afternoon', 'Evening'])

# Season
df['season'] = df['appt_month'].map({12:0, 1:0, 2:0, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2, 9:3, 10:3, 11:3})

# Lead time
df['lead_time_days'] = (df['startDate'] - df['createOn']).dt.total_seconds() / 86400
df['was_updated'] = df['updatedOn'].notna().astype(int)

# Days since last payment
df['days_since_last_payment'] = (df['startDate'] - df['lastPaymentDate']).dt.days

# Insurance coverage
df['insurance_coverage_days'] = (df['startDate'] - df['insurance_coverageStartDate']).dt.days
df['secondary_insurance_active'] = ((df['secondaryInsurance_coverageStartDate'].notna()) & 
                                     ((df['secondaryInsurance_terminationDate'].isna()) | 
                                      (df['startDate'] < df['secondaryInsurance_terminationDate']))).astype(int)

# ENHANCEMENT 1: Holiday flags (US Federal Holidays)
from pandas.tseries.holiday import USFederalHolidayCalendar
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=df['startDate'].min(), end=df['startDate'].max())
df['is_holiday'] = df['startDate'].dt.normalize().isin(holidays).astype(int)

# ENHANCEMENT 1: Peak day flag (Monday mornings 8-11 AM)
df['is_peak_day'] = ((df['appt_weekday'] == 0) & (df['appt_hour'].between(8, 11))).astype(int)

print("Temporal features + holidays + peak days created")

Temporal features + holidays + peak days created


## 4. Provider & Patient Interaction Features + Rolling Patient Metrics

In [19]:
df = df.sort_values('startDate').reset_index(drop=True)

# Provider metrics
provider_counts = df.groupby('providerId').size().to_dict()
df['provider_total_appts'] = df['providerId'].map(provider_counts)

provider_avg_duration = df.groupby('providerId')['duration'].mean().to_dict()
df['provider_avg_duration'] = df['providerId'].map(provider_avg_duration)

# Patient metrics
patient_counts = df.groupby('patientId').size().to_dict()
df['patient_total_appts'] = df['patientId'].map(patient_counts)

# Patient-Provider history
patient_provider_counts = df.groupby(['patientId', 'providerId']).size().to_dict()
df['patient_provider_history'] = df.apply(lambda x: patient_provider_counts.get((x['patientId'], x['providerId']), 0), axis=1)

# Cancellation rates
provider_cancel = df[df['appointmentStatus'].isin(['Rescheduled', 'Cancelled'])].groupby('providerId').size() / df.groupby('providerId').size()
df['provider_cancel_rate'] = df['providerId'].map(provider_cancel).fillna(0)

patient_cancel = df[df['appointmentStatus'].isin(['Rescheduled', 'Cancelled'])].groupby('patientId').size() / df.groupby('patientId').size()
df['patient_cancel_rate'] = df['patientId'].map(patient_cancel).fillna(0)

# ENHANCEMENT 4: Rolling patient metrics (7 and 30 days)
df['patient_7day_appts'] = df.groupby('patientId').cumcount().rolling(window=7, min_periods=1).sum().reset_index(0, drop=True)
df['patient_30day_appts'] = df.groupby('patientId').cumcount().rolling(window=30, min_periods=1).sum().reset_index(0, drop=True)

df['is_cancelled'] = df['appointmentStatus'].isin(['Rescheduled', 'Cancelled']).astype(int)
df['patient_7day_cancel'] = df.groupby('patientId')['is_cancelled'].transform(lambda x: x.rolling(window=7, min_periods=1).sum())
df['patient_30day_cancel'] = df.groupby('patientId')['is_cancelled'].transform(lambda x: x.rolling(window=30, min_periods=1).sum())

print("Interaction features + rolling patient metrics created")

Interaction features + rolling patient metrics created


## 5. Vectorized Conflict Detection & Provider Utilization

In [20]:
# ENHANCEMENT 2: Vectorized overlap detection
df_sorted = df.sort_values(['providerId', 'startDate']).reset_index(drop=True)
df_sorted['prev_endDate'] = df_sorted.groupby('providerId')['endDate'].shift(1)
df_sorted['has_overlap'] = ((df_sorted['startDate'] < df_sorted['prev_endDate']) & 
                             (df_sorted['providerId'] == df_sorted['providerId'].shift(1))).astype(int)
df['has_overlap'] = df_sorted['has_overlap'].values

# Rolling provider utilization
df['provider_7day_util'] = df.groupby('providerId')['duration'].transform(lambda x: x.rolling(window=7, min_periods=1).sum())
df['provider_30day_util'] = df.groupby('providerId')['duration'].transform(lambda x: x.rolling(window=30, min_periods=1).sum())

print("Vectorized conflict detection and utilization metrics created")

Vectorized conflict detection and utilization metrics created


## 6. Insurance & Payment Features

In [21]:
df['has_primary_insurance'] = (df['hasInsurance'] == 'yes').astype(int)
df['has_secondary_insurance'] = (df['hasSecondaryInsurance'] == 'yes').astype(int)
df['has_dual_insurance'] = ((df['has_primary_insurance'] == 1) & (df['has_secondary_insurance'] == 1)).astype(int)

df['is_medicare'] = df['insurance_insuranceType'].str.contains('Medicare', case=False, na=False).astype(int)
df['is_medicaid'] = df['insurance_insuranceType'].str.contains('Medicaid|Medi-Cal', case=False, na=False).astype(int)
df['is_hmo'] = df['insurance_insuranceType'].str.contains('HMO', case=False, na=False).astype(int)

df['is_paid'] = (df['paymentStatus'] == 'Paid').astype(int)
df['has_copay'] = df['insurance_coPayAmount'].notna().astype(int)

patient_avg_copay = df.groupby('patientId')['insurance_coPayAmount'].mean().to_dict()
df['patient_avg_copay'] = df['patientId'].map(patient_avg_copay)

print("Insurance and payment features created")

Insurance and payment features created


## 7. Visit Type & Enhanced NLP Features

In [22]:
df['duration_category'] = pd.cut(df['duration'], bins=[0, 15, 30, 60], labels=['Short', 'Medium', 'Long'])

df['is_home_visit'] = df['visitReasonName'].str.contains('Home Visit', case=False, na=False).astype(int)
df['is_new_patient'] = df['visitReasonName'].str.contains('New Patient', case=False, na=False).astype(int)
df['is_established_patient'] = df['visitReasonName'].str.contains('Established Patient', case=False, na=False).astype(int)
df['is_telehealth'] = df['isTelehealth'].astype(int)

df['is_wellness'] = df['visitReasonmsg'].str.contains('Wellness|Annual', case=False, na=False).astype(int)
df['is_followup'] = df['visitReasonmsg'].str.contains('Follow', case=False, na=False).astype(int)
df['is_lab'] = df['visitReasonmsg'].str.contains('Lab|Test', case=False, na=False).astype(int)
df['is_emergency'] = df['visitReasonmsg'].str.contains('ER|Emergency', case=False, na=False).astype(int)

df['note_length'] = df['note'].fillna('').str.len()

# ENHANCEMENT 5: TF-IDF features from note and visitReasonmsg
tfidf_note = TfidfVectorizer(max_features=50, stop_words='english')
note_features = tfidf_note.fit_transform(df['note'].fillna('')).toarray()
note_df = pd.DataFrame(note_features, columns=[f'note_tfidf_{i}' for i in range(note_features.shape[1])])
df = pd.concat([df.reset_index(drop=True), note_df], axis=1)

tfidf_reason = TfidfVectorizer(max_features=50, stop_words='english')
reason_features = tfidf_reason.fit_transform(df['visitReasonmsg'].fillna('')).toarray()
reason_df = pd.DataFrame(reason_features, columns=[f'reason_tfidf_{i}' for i in range(reason_features.shape[1])])
df = pd.concat([df.reset_index(drop=True), reason_df], axis=1)

print("Visit type and enhanced NLP features created")

Visit type and enhanced NLP features created


## 8. Handle Missing Values + Room/Location

In [23]:
# ENHANCEMENT 6: Handle roomNumber and serviceLocationId
df['roomNumber'] = df['roomNumber'].fillna('Unassigned')
df['serviceLocationId'] = df['serviceLocationId'].fillna('Unknown')

# Numeric columns
numeric_cols = ['paidAmount', 'insurance_coPayAmount', 'secondaryInsurance_coPayAmount', 'lead_time_days', 
                'days_since_last_payment', 'insurance_coverage_days', 'patient_avg_copay', 'note_length']
for col in numeric_cols:
    if col in df.columns:
        df[col] = df.groupby('providerId')[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median())

# Categorical columns
cat_cols = ['visitReasonColorInCalender', 'insurance_insuranceType', 'insurance_insuranceCompany', 
            'insurance_medicalGroup', 'insurance_pcp', 'serviceLocationName']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

df['patient_age'] = df['patient_age'].fillna(df['patient_age'].median())

print(f"Missing values handled. Remaining nulls: {df.isnull().sum().sum()}")

Missing values handled. Remaining nulls: 34856


## 9. Encode Categorical Variables + Frequency Encoding

In [24]:
# Label encoding
le_sex = LabelEncoder()
df['sex_encoded'] = le_sex.fit_transform(df['sex'])

le_status = LabelEncoder()
df['appt_status_encoded'] = le_status.fit_transform(df['appointmentStatus'])

le_contact = LabelEncoder()
df['contact_method_encoded'] = le_contact.fit_transform(df['preferredContactMethod'])

le_provider = LabelEncoder()
df['provider_encoded'] = le_provider.fit_transform(df['providerId'])

le_patient = LabelEncoder()
df['patient_encoded'] = le_patient.fit_transform(df['patientId'])

le_visit = LabelEncoder()
df['visit_reason_encoded'] = le_visit.fit_transform(df['visitReasonName'])

le_location = LabelEncoder()
df['location_encoded'] = le_location.fit_transform(df['serviceLocationName'])

# ENHANCEMENT 3: Frequency encoding for high-cardinality features
freq_cols = ['insurance_medicalGroup', 'providerName', 'serviceLocationName']
for col in freq_cols:
    if col in df.columns:
        freq_map = df[col].value_counts(normalize=True).to_dict()
        df[f'{col}_freq'] = df[col].map(freq_map)

# One-hot encoding
df = pd.get_dummies(df, columns=['time_slot', 'duration_category'], prefix=['slot', 'dur'], drop_first=False)

print("Encoding + frequency encoding completed")

Encoding + frequency encoding completed


## 10. Select ML Features

In [25]:
ml_features = [
    'patient_age', 'sex_encoded', 'patient_total_appts', 'patient_encoded', 'patient_cancel_rate',
    'patient_7day_appts', 'patient_30day_appts', 'patient_7day_cancel', 'patient_30day_cancel',
    'provider_encoded', 'provider_total_appts', 'provider_avg_duration', 'provider_cancel_rate',
    'provider_7day_util', 'provider_30day_util',
    'duration', 'appt_weekday', 'appt_hour', 'appt_month', 'appt_day', 'appt_quarter',
    'is_weekend', 'is_holiday', 'is_peak_day', 'season', 'lead_time_days', 'was_updated', 
    'visit_reason_encoded', 'location_encoded',
    'patient_provider_history', 'has_overlap',
    'has_primary_insurance', 'has_secondary_insurance', 'has_dual_insurance',
    'is_medicare', 'is_medicaid', 'is_hmo', 'secondary_insurance_active',
    'insurance_coverage_days', 'has_copay', 'patient_avg_copay',
    'is_home_visit', 'is_new_patient', 'is_established_patient', 'is_telehealth',
    'is_wellness', 'is_followup', 'is_lab', 'is_emergency',
    'is_paid', 'paidAmount', 'days_since_last_payment',
    'contact_method_encoded', 'note_length',
    'insurance_medicalGroup_freq', 'providerName_freq', 'serviceLocationName_freq'
]

# Add one-hot and TF-IDF columns
slot_cols = [col for col in df.columns if col.startswith('slot_')]
dur_cols = [col for col in df.columns if col.startswith('dur_')]
note_tfidf_cols = [col for col in df.columns if col.startswith('note_tfidf_')]
reason_tfidf_cols = [col for col in df.columns if col.startswith('reason_tfidf_')]

ml_features.extend(slot_cols)
ml_features.extend(dur_cols)
ml_features.extend(note_tfidf_cols)
ml_features.extend(reason_tfidf_cols)

target = 'appt_status_encoded'
df_ml = df[ml_features + [target]].copy()

print(f"ML-Ready Dataset Shape: {df_ml.shape}")
print(f"Total Features: {len(ml_features)}")

ML-Ready Dataset Shape: (1000, 93)
Total Features: 92


## 11. Feature Scaling

In [26]:
numeric_features = ['patient_age', 'duration', 'lead_time_days', 'provider_total_appts', 'patient_total_appts',
                   'provider_avg_duration', 'patient_provider_history', 'paidAmount', 'provider_7day_util',
                   'provider_30day_util', 'days_since_last_payment', 'insurance_coverage_days', 
                   'patient_avg_copay', 'note_length', 'provider_cancel_rate', 'patient_cancel_rate',
                   'patient_7day_appts', 'patient_30day_appts', 'patient_7day_cancel', 'patient_30day_cancel']

scaler = StandardScaler()
df_ml[numeric_features] = scaler.fit_transform(df_ml[numeric_features])

print("Feature scaling completed")

Feature scaling completed


## 12. Save Processed Data

In [27]:
df_ml.to_csv('../dataset/processed/ml_ready_appointments.csv', index=False)
print("ML-ready dataset saved")

df.to_csv('../dataset/processed/full_processed_appointments.csv', index=False)
print("Full processed dataset saved")

ML-ready dataset saved
Full processed dataset saved


## 13. Summary Statistics

In [28]:
print("="*60)
print("ENHANCED ML-READY DATASET SUMMARY")
print("="*60)
print(f"Total Records: {len(df_ml)}")
print(f"Total Features: {len(ml_features)}")
print(f"Target Variable: {target}")
print(f"\nEnhancements Applied:")
print(f"  1. Holiday & Peak Day Flags")
print(f"  2. Vectorized Overlap Detection")
print(f"  3. Frequency Encoding (3 features)")
print(f"  4. Rolling Patient Metrics (4 features)")
print(f"  5. TF-IDF NLP Features (100 features)")
print(f"  6. Room/Location Handling")
print(f"\nFeature Categories:")
print(f"  Patient: 9 | Provider: 6 | Appointment: 15")
print(f"  Interaction: 2 | Insurance: 11 | Visit Type: 8")
print(f"  Payment: 3 | Contact: 2 | Frequency: 3")
print(f"  Time Slots: {len(slot_cols)} | Duration: {len(dur_cols)}")
print(f"  NLP Features: {len(note_tfidf_cols) + len(reason_tfidf_cols)}")
print(f"\nTarget Distribution:\n{df_ml[target].value_counts()}")

ENHANCED ML-READY DATASET SUMMARY
Total Records: 1000
Total Features: 92
Target Variable: appt_status_encoded

Enhancements Applied:
  1. Holiday & Peak Day Flags
  2. Vectorized Overlap Detection
  3. Frequency Encoding (3 features)
  4. Rolling Patient Metrics (4 features)
  5. TF-IDF NLP Features (100 features)
  6. Room/Location Handling

Feature Categories:
  Patient: 9 | Provider: 6 | Appointment: 15
  Interaction: 2 | Insurance: 11 | Visit Type: 8
  Payment: 3 | Contact: 2 | Frequency: 3
  Time Slots: 4 | Duration: 3
  NLP Features: 28

Target Distribution:
appt_status_encoded
1    751
0    172
2     77
Name: count, dtype: int64
